In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
ROOT = Path.cwd().parent
RAW = ROOT / "data" / "generated data"
PNL = ROOT / "data" / "pnl"
VAR = ROOT/ "data" / "variance"

for d in (RAW, PNL, VAR): d.mkdir(parents = True, exist_ok = True)

In [25]:
#Generate master data PRODUCTS REGIONS ACCOUNTS and dates
PRODUCTS = pd.DataFrame({
    "product_id" : ["P01", "P02", "P03", "P04"],
    "product_name" : ["Alpha", "Beta", "Gamma", "Delta"],
    "base_price" : [100, 120, 200, 300],
})
REGIONS = pd.DataFrame({
    "region_id" : ["NAM", "EU", "APAC", "OTHER"],
    "region_name" : ["North America", "Europe", "Asia, Pacific", "Other"],
})
ACCOUNTS = pd.DataFrame({
    "account_id" : ["REV", "COGS", "PAYROLL", "MARKETING", "IT", "TRAVEL"],
    "account_name" : ["Revenue", "COGS", "Payroll", "Marketing", "IT", "Travel"],
    "account_type" : ["Revenue", "COGS", "OPEX", "OPEX", "OPEX", "OPEX"]
})

def _dates():
    return pd.date_range("2024-01-01", "2026-12-31", freq = "MS")

CLOSE_MONTH = pd.Timestamp("2026-10-01")
FCST_VERSIONS = {
    "v1": pd.Timestamp("2026-03-01"),
    "v2": pd.Timestamp("2026-06-01"),
    "v3": pd.Timestamp("2026-09-01"),
}

In [26]:
#Generate business scenarios
def _scenario_factor(date, product, region, rng):
    vf = 1.0
    pf = 1.0
    cf = 1.0
    if(pd.Timestamp("2026-04-01") <= date and pd.Timestamp("2026-06-01") >= date ):
        if(product == "P01" and region == "EU"):
            vf *= 0.85
        if(product == "P02" and region == "NAM"):
            pf *= 1.06
        cf *= 1.07
    if(pd.Timestamp("2026-05-01") <= date and region == "EU"):
        vf *= 0.97
    return vf, pf, cf

In [27]:
def generate_all(seed = 42):
    rng = np.random.default_rng(seed)
    dates = _dates()
    
    #Export master data
    PRODUCTS.to_csv(RAW/"dim_products.csv", index=False)
    REGIONS.to_csv(RAW/"dim_regions.csv", index = False)
    ACCOUNTS.to_csv(RAW/"dim_accounts.csv", index = False)
    
    #Generate 100 random customer number and assign them randomly to each region with weight
    CUSTOMERS = pd.DataFrame({
        "customer_id" : [f"C{i:03d}" for i in range(1, 101)],
        "region_id" : rng.choice(REGIONS.region_id, 100, p = [0.35, 0.30, 0.25, 0.10]),
    })
    
    #Export master data
    CUSTOMERS.to_csv(RAW/"dim_customers.csv", index = False)
    
    #Generate transaction data - Actuals
    ACTUALS = []
    BUDGETS = []
    
    #Generate base data for each product and region
    BASE = {(p, r): rng.uniform(500, 1800)
           for p in PRODUCTS.product_id for r in REGIONS.region_id}
    PRICE = PRODUCTS.set_index("product_id").base_price.to_dict()
    
    #Create records in each date a product_id and a region_id, calculate the volume, price, dicount, revenue
    for date in dates:
        season = 1 + 0.08 * np.sin(2*np.pi*(date.month-1)/12) #Add seasonality to volume
        for (pid, region), base in BASE.items():
            #Budget: the plan, no noise, no scenario
            bud_volume = base * season
            bud_price = PRICE[pid]
            bud_disc = 0.06
            BUDGETS.append([date, pid, region, bud_volume, bud_price, bud_disc, bud_volume * bud_price * (1-bud_disc)])
            
            #Actuals: base + noise + scenario
            vf, pf, cf = _scenario_factor(date, pid, region, rng)
            if date > CLOSE_MONTH:
                continue
            act_volume = base * season * (1+rng.normal(0, 0.06)) * vf #Add randomization to volume
            act_price = PRICE[pid] * (1+rng.normal(0, 0.025)) * pf
            act_disc = np.clip(rng.normal(0.06, 0.02), 0, 0.15)
            ACTUALS.append([date, pid, region, act_volume, act_price, act_disc, act_volume * act_price * (1-act_disc), cf])
            
    cols = ["date", "product_id", "region_id", "volume", "price", "discount", "revenue"]
    SALES = pd.DataFrame(ACTUALS, columns = cols + ["cost_factor"])
    BUDGET = pd.DataFrame(BUDGETS, columns = cols)
    BUDGET["cost_factor"] = 1.0
 
    #Export Sales actuals and budget data
    SALES.to_csv(RAW/"fact_sales_actual.csv", index = False)
    BUDGET.to_csv(RAW/"fact_sales_budget.csv", index = False)
        
    #Transform SALES data to financial reporting data
    #Aggregate SALES/revenue by month and region, create monthly Opex
    OPEX_ANNUAL = {"PAYROLL": 850_000, "MARKETING": 220_000, "IT": 110_000, "TRAVEL": 75000}
    
    def to_finance(sales, cogs_rate, opex_scale, noise=False):
        agg = (sales.assign(REVENUE = sales.revenue, COGS = -sales.revenue * cogs_rate * sales.cost_factor)
               .groupby(["date", "region_id"])[["REVENUE", "COGS"]].sum())
        for acct, annual in OPEX_ANNUAL.items():
            f = rng.normal(1, 0.05, len(agg)) if noise else 1
            agg[acct] = -annual / 12 * opex_scale * f
        return agg.reset_index().melt(id_vars = ["date", "region_id"], var_name = "account_id", value_name = "value")
    
    FINANCE_ACT = to_finance(SALES, 0.47, 1.0, noise=True)
    FINANCE_BUD = to_finance(BUDGET, 0.46, 0.97)  
    
    #Export financial actuals and budget data
    FINANCE_ACT.to_csv(RAW/"fact_finance_actual.csv", index = False)
    FINANCE_BUD.to_csv(RAW/"fact_finance_budget.csv", index = False)
    
    def rolling_forecast(actual, plan, cutoff):
        #Before cutoff date use actual data, else use plan data 
        return pd.concat([actual[actual.date <= cutoff], plan[plan.date > cutoff]], ignore_index = True)
    
    for version, cutoff in FCST_VERSIONS.items():
        rolling_forecast(SALES, BUDGET, cutoff).to_csv(RAW/f"fact_sales_forecast_{version}.csv", index = False)
        rolling_forecast(FINANCE_ACT, FINANCE_BUD, cutoff).to_csv(RAW/f"fact_finance_forecast_{version}.csv", index = False)
        
    #Create operational KPI
    #Calculate volume, price, revenue by date, region, and product
    OPERATION = SALES.groupby(["date", "region_id", "product_id"], as_index = False).agg(
    volume = ("volume", "sum"), average_price = ("price", "mean"), revenue = ("revenue", "sum"))
    
    #Add customer count to operation data
    OPERATION["customer_count"] = rng.integers(40, 100, len(OPERATION))
    
    #Export operation data
    OPERATION.to_csv(RAW/"fact_operation.csv", index = False)

In [28]:
generate_all()